In [1]:
# Install necessary libraries (already done in your original notebooks, but good to include for completeness)
!pip install sentence-transformers pytrec_eval pandas tqdm

# Imports
import os
import json
import gzip
import torch
from tqdm import tqdm
from sentence_transformers import CrossEncoder, util
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from collections import defaultdict
import numpy as np
import pytrec_eval
import operator
import logging

# Setup logging
logger = logging.getLogger()
logger.setLevel(logging.INFO) # Changed to INFO to reduce verbosity during download
logging.basicConfig(format='%(asctime)s - %(message)s',datefmt='%Y-%m-%d %H:%M:%S')


# Mount Google Drive to access models and expanded queries
from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/"

# Define paths to your saved models and expanded queries
minilm_model_save_path = base_path + "finetuned_models/cross-encoder-ms-marco-MiniLM-L-2-v2"
distilroberta_model_save_path = base_path + "finetuned_models/cross-encoder-distilroberta-base"
tinybert_model_save_path = base_path + "finetuned_models/cross-encoder-ms-marco-tinybert-l-2-v2"
expanded_queries_path = base_path + "expanded_queries.json" # Path to your saved JSON file

# --- Data Loading ---
# Load MSMARCO Queries and Candidate Docs (using the TREC DL 2019 test set)
data_folder = 'trec2019-data'
os.makedirs(data_folder, exist_ok=True)

# Read test queries
queries = {}
queries_filepath = os.path.join(data_folder, 'msmarco-test2019-queries.tsv.gz')
if not os.path.exists(queries_filepath):
    logging.info(f"Download {os.path.basename(queries_filepath)}")
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz', queries_filepath)

with gzip.open(queries_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query

# Read which passages are relevant (TREC DL 2019 qrels)
relevant_docs = defaultdict(lambda: defaultdict(int))
qrels_filepath = os.path.join(data_folder, '2019qrels-pass.txt')

if not os.path.exists(qrels_filepath):
    logging.info(f"Download {os.path.basename(qrels_filepath)}")
    util.http_get('https://trec.nist.gov/data/deep/2019qrels-pass.txt', qrels_filepath)

with open(qrels_filepath) as fIn:
    for line in fIn:
        qid, _, pid, score = line.strip().split()
        score = int(score)
        if score > 0: # Only consider truly relevant documents
            relevant_docs[qid][pid] = score

# Get the list of query IDs that were used in the LLM expansion
# This should match the first 43 QIDs from the queries file based on your generation notebook
llm_expanded_qids = list(queries.keys())[:43] # Assuming the LLM expanded the first 43 queries

# Read the top 1000 passages for the relevant queries
passage_cand = {}
passage_filepath = os.path.join(data_folder, 'msmarco-passagetest2019-top1000.tsv.gz')

if not os.path.exists(passage_filepath):
    logging.info(f"Download {os.path.basename(passage_filepath)}")
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz', passage_filepath)

# Load only candidates for the queries we care about (the 43 expanded ones)
with gzip.open(passage_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, pid, _, passage = line.strip().split("\t")
        if qid in llm_expanded_qids: # Filter for the 43 QIDs
             if qid not in passage_cand:
                passage_cand[qid] = []
             passage_cand[qid].append([pid, passage])


logging.info(f"Loaded {len(queries)} original queries.")
logging.info(f"Loaded {len(relevant_docs)} queries with relevance judgments.")
logging.info(f"Loaded candidate passages for {len(passage_cand)} queries.")
logging.info(f"Evaluating on the first {len(llm_expanded_qids)} queries used for LLM expansion.")


# Load the LLM-expanded queries
expanded_queries = {}
try:
    with open(expanded_queries_path, "r") as f:
        expanded_queries = json.load(f)
    logging.info(f"Loaded {len(expanded_queries)} expanded queries from {expanded_queries_path}.")
except FileNotFoundError:
    logging.error(f"Error: Expanded queries file not found at {expanded_queries_path}")
    # Exit or handle the error appropriately if the file is missing
    exit()

# Filter relevant_docs and passage_cand to only include the expanded queries
relevant_docs_expanded_subset = {qid: docs for qid, docs in relevant_docs.items() if qid in expanded_queries}
passage_cand_expanded_subset = {qid: cand for qid, cand in passage_cand.items() if qid in expanded_queries}

logging.info(f"Filtered relevance judgments for {len(relevant_docs_expanded_subset)} expanded queries.")
logging.info(f"Filtered candidate passages for {len(passage_cand_expanded_subset)} expanded queries.")


# Define evaluation function
def evaluate_cross_encoder(model, queries_dict, candidate_passages, relevance_judgments, eval_name):
    """
    Evaluates a cross-encoder model on a given set of queries and candidates.

    Args:
        model: The loaded Sentence-Transformer CrossEncoder model.
        queries_dict (dict): Dictionary of query IDs to query texts.
        candidate_passages (dict): Dictionary of query IDs to list of [pid, passage_text].
        relevance_judgments (dict): Dictionary of query IDs to dictionary of passage IDs to scores.
        eval_name (str): Name for this evaluation run (e.g., "Original Query", "Expanded Query").

    Returns:
        dict: Dictionary of evaluation metrics.
    """
    logging.info(f"Starting evaluation with {eval_name} for {len(queries_dict)} queries.")

    run = {}
    # Use tqdm only for the queries being evaluated
    for qid, query_text in tqdm(queries_dict.items(), desc=f"Predicting with {eval_name}"):
        if qid not in candidate_passages:
            logging.warning(f"No candidate passages found for query {qid}. Skipping.")
            continue

        cand = candidate_passages[qid]
        pids = [c[0] for c in cand]
        corpus_sentences = [c[1] for c in cand]

        cross_inp = [[query_text, sent] for sent in corpus_sentences]

        # Truncate input to fit model's max length if necessary
        # (CrossEncoder handles this internally, but it's good practice to be aware)
        # Max length for most models is 512 tokens. Long queries/passages will be truncated.

        if model.config.num_labels > 1:
            # Cross-Encoder predicts more than 1 score (e.g., [non-relevant, relevant])
            # We use the score for the 'relevant' label (usually index 1) and apply softmax
            cross_scores = model.predict(cross_inp, apply_softmax=True)[:, 1].tolist()
        else:
            # Cross-Encoder predicts a single score (e.g., relevance score)
            cross_scores = model.predict(cross_inp).tolist()

        # Store scores in sparse format
        run[qid] = {}
        for idx, pid in enumerate(pids):
            run[qid][pid] = float(cross_scores[idx])

    # Sort candidates by score for evaluation metrics
    sorted_run = {}
    for qid in run.keys():
         sorted_run[qid] = dict(sorted(run[qid].items(), key=operator.itemgetter(1), reverse=True))


    # Use pytrec_eval for evaluation
    evaluator = pytrec_eval.RelevanceEvaluator(relevance_judgments, {'ndcg_cut.10', 'recall_100', 'map_cut.1000'})
    scores = evaluator.evaluate(sorted_run)

    avg_ndcg = np.mean([ele.get("ndcg_cut_10", 0) for ele in scores.values()]) * 100
    avg_recall = np.mean([ele.get("recall_100", 0) for ele in scores.values()]) * 100
    avg_map = np.mean([ele.get("map_cut_1000", 0) for ele in scores.values()]) * 100

    return {
        "name": eval_name,
        "NDCG@10": avg_ndcg,
        "Recall@100": avg_recall,
        "MAP@1000": avg_map
    }

# --- Load Models ---
logging.info("Loading cross-encoder models...")
minilm_model = CrossEncoder(minilm_model_save_path)
distilroberta_model = CrossEncoder(distilroberta_model_save_path)
tinybert_model = CrossEncoder(tinybert_model_save_path)
logging.info("Models loaded.")

# --- Filter Original Queries to Match Expanded Subset ---
original_queries_expanded_subset = {qid: queries[qid] for qid in llm_expanded_qids}

# --- Run Evaluations ---
evaluation_results = {}

# Evaluate MiniLM
evaluation_results['MiniLM_Original'] = evaluate_cross_encoder(
    minilm_model, original_queries_expanded_subset, passage_cand_expanded_subset, relevant_docs_expanded_subset, "MiniLM (Original Query)"
)
evaluation_results['MiniLM_Expanded'] = evaluate_cross_encoder(
    minilm_model, expanded_queries, passage_cand_expanded_subset, relevant_docs_expanded_subset, "MiniLM (Expanded Query)"
)

# Evaluate Distilroberta
evaluation_results['Distilroberta_Original'] = evaluate_cross_encoder(
    distilroberta_model, original_queries_expanded_subset, passage_cand_expanded_subset, relevant_docs_expanded_subset, "Distilroberta (Original Query)"
)
evaluation_results['Distilroberta_Expanded'] = evaluate_cross_encoder(
    distilroberta_model, expanded_queries, passage_cand_expanded_subset, relevant_docs_expanded_subset, "Distilroberta (Expanded Query)"
)

# Evaluate Tinybert
evaluation_results['Tinybert_Original'] = evaluate_cross_encoder(
    tinybert_model, original_queries_expanded_subset, passage_cand_expanded_subset, relevant_docs_expanded_subset, "Tinybert (Original Query)"
)
evaluation_results['Tinybert_Expanded'] = evaluate_cross_encoder(
    tinybert_model, expanded_queries, passage_cand_expanded_subset, relevant_docs_expanded_subset, "Tinybert (Expanded Query)"
)


# --- Report Results ---
print("\\n--- Evaluation Results ---")
for key, metrics in evaluation_results.items():
    print(f"\\n{metrics['name']}:")
    print(f"  Queries: {len(original_queries_expanded_subset)}") # Report on the size of the subset evaluated
    print(f"  NDCG@10: {metrics['NDCG@10']:.2f}")
    print(f"  Recall@100: {metrics['Recall@100']:.2f}")
    print(f"  MAP@1000: {metrics['MAP@1000']:.2f}")

# --- END OF FILE evaluate_query_expansion.ipynb ---

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.4 MB/s eta 0:00:00
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp311-cp311-linux_x86_64.whl size=308669 sha256=213c3d6b64b80f6b7c789f239965fb980c56526288820d27bade3be

INFO:root:Download msmarco-test2019-queries.tsv.gz


Mounted at /content/drive


  0%|          | 0.00/4.28k [00:00<?, ?B/s]

INFO:root:Download 2019qrels-pass.txt


...

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Predicting with Tinybert (Expanded Query):  98%|█████████▊| 42/43 [00:30<00:00,  1.33it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Predicting with Tinybert (Expanded Query): 100%|██████████| 43/43 [00:31<00:00,  1.38it/s]

\n--- Evaluation Results ---
\nMiniLM (Original Query):
  Queries: 43
  NDCG@10: 55.52
  Recall@100: 37.28
  MAP@1000: 32.66
\nMiniLM (Expanded Query):
  Queries: 43
  NDCG@10: 33.93
  Recall@100: 27.04
  MAP@1000: 18.03
\nDistilroberta (Original Query):
  Queries: 43
  NDCG@10: 53.26
  Recall@100: 37.46
  MAP@1000: 32.77
\nDistilroberta (Expanded Query):
  Queries: 43
  NDCG@10: 23.26
  Recall@100: 26.19
  MAP@1000: 14.40
\nTinybert (Original Query):
  Queries: 43
  NDCG@10: 61.84
  Recall@100: 39.50
  MAP@1000: 35.88
\nTinybert (Expanded Query):
  Queries: 43
  NDCG@10: 28.05
  Recall@100: 28.30
  MAP@1000: 18.52
